# FacemoodClassifier — Depression-related facial expression classification

**Team:** Sara Bedulla, Kejdi Shahu

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaraBedulla/FacemoodClassifier/blob/main/notebooks/FacemoodClassifier_colab.ipynb)

This notebook runs the full pipeline from the [project repository](https://github.com/SaraBedulla/FacemoodClassifier):
data preparation → exploratory analysis → training (CNN from scratch, MobileNetV2, ResNet50)
→ test-set evaluation → comparison.

It works in two places with the same code:
- **Google Colab**: clones the repo and downloads FER-2013 and RAF-DB automatically. Use *Runtime → Change runtime type → T4 GPU*.
- **Locally** (PyCharm / Jupyter): uses the repo you already have and the dataset in `data/raw/`.

## 1. Setup

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not Path("FacemoodClassifier").exists():
        !git clone https://github.com/SaraBedulla/FacemoodClassifier.git
    %cd FacemoodClassifier
    # Colab already has TensorFlow, scikit-learn, pandas, matplotlib and pillow.
    !pip install -q kagglehub
else:
    # Running locally from notebooks/: move to the project root.
    if Path.cwd().name == "notebooks":
        os.chdir("..")

print("Project root:", Path.cwd())

In [ ]:
import tensorflow as tf
print("TensorFlow", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "none (training will be slow)")

## 2. Dataset

**FER-2013** ([Kaggle](https://www.kaggle.com/datasets/msambare/fer2013)): 35,887 grayscale 48×48 face images,
7 emotions (angry, disgust, fear, happy, sad, surprise, neutral). A stratified 10% of the training set is held out
for validation, giving 25,838 train / 2,871 validation / 7,178 test images.

In [ ]:
local_source = Path("data/raw/fer2013")
if local_source.exists():
    source = local_source
else:
    import kagglehub
    source = Path(kagglehub.dataset_download("msambare/fer2013"))
print("Dataset source:", source)

In [ ]:
!python -m src.data.prepare --dataset fer2013 --source "{source}"

## 3. Exploratory data analysis

In [ ]:
!python -m src.data.eda --dataset fer2013

In [ ]:
from IPython.display import Image, display

eda_dir = Path("results/figures/eda/fer2013")
for name in ["class_distribution_emotion7", "class_distribution_affect3", "samples", "mean_faces"]:
    print(name)
    display(Image(filename=str(eda_dir / f"{name}.png"), width=700))

## 4. Training — CNN from scratch

Two label schemes: `emotion7` (all 7 emotions) and `affect3` (negative / neutral / positive).
Each run keeps the weights with the lowest validation loss (early stopping, patience 10).

In [ ]:
!python -m src.training.train --model scratch_cnn --labels emotion7

In [ ]:
!python -m src.training.train --model scratch_cnn --labels affect3

## 5. Training — transfer learning (MobileNetV2, ResNet50)

ImageNet-pretrained backbones with a new classification head. The 48×48 grayscale faces are resized to
224×224 and copied to 3 channels. Each run trains in two phases:
1. **Head**: backbone frozen, only the new head trains (learning rate 1e-3, up to 10 epochs).
2. **Fine-tune**: the top backbone stage is unfrozen (MobileNetV2: last 3 blocks, ResNet50: conv5)
   and trained together with the head at learning rate 1e-5, with early stopping.

At 224×224 these runs are much slower than the scratch CNN (ResNet50 most of all), so use a GPU runtime.

In [ ]:
!python -m src.training.train --model mobilenetv2 --labels emotion7
!python -m src.training.train --model mobilenetv2 --labels affect3

In [ ]:
!python -m src.training.train --model resnet50 --labels emotion7
!python -m src.training.train --model resnet50 --labels affect3

## 6. Second dataset: RAF-DB

**RAF-DB** ([Kaggle copy](https://www.kaggle.com/datasets/shuvoalok/raf-db-dataset)): 15,339 aligned colour 100×100 face
images with the same 7 emotions, each labelled by about 40 annotators. A stratified 10% of the training set is held out
for validation, giving 11,043 train / 1,228 validation / 3,068 test images. All three models are trained on it with the
same settings as on FER-2013, except the scratch CNN on `emotion7`, which uses a smaller learning rate and a longer budget.

In [ ]:
local_raf = Path("data/raw/rafdb")
if local_raf.exists():
    raf_source = local_raf
else:
    import kagglehub
    raf_source = Path(kagglehub.dataset_download("shuvoalok/raf-db-dataset"))
print("Dataset source:", raf_source)

!python -m src.data.prepare --dataset rafdb --source "{raf_source}"
!python -m src.data.eda --dataset rafdb

In [ ]:
!python -m src.training.train --dataset rafdb --model scratch_cnn --labels emotion7 --lr 0.0003 --epochs 100 --patience 20
!python -m src.training.train --dataset rafdb --model scratch_cnn --labels affect3

In [ ]:
for model in ["mobilenetv2", "resnet50"]:
    for labels in ["emotion7", "affect3"]:
        !python -m src.training.train --dataset rafdb --model {model} --labels {labels}

## 7. Training and validation accuracy / loss

For the transfer models, the dashed line marks where fine-tuning starts.

In [ ]:
import json
import pandas as pd

runs = sorted(p for p in Path("results/runs").iterdir() if p.is_dir())
summaries = [json.loads((r / "train_summary.json").read_text()) for r in runs if (r / "train_summary.json").exists()]
display(pd.DataFrame(summaries))

for r in runs:
    curves = r / "training_curves.png"
    if curves.exists():
        print(r.name)
        display(Image(filename=str(curves), width=800))

## 8. Test-set evaluation

Accuracy, macro-F1 (the classes are imbalanced), per-class recall, a confusion matrix and sample predictions
(correct and incorrect) for every run. Runs that were already evaluated are skipped.

In [ ]:
!python -m src.evaluation.evaluate

In [ ]:
for r in runs:
    if (r / "test_metrics.json").exists():
        metrics = json.loads((r / "test_metrics.json").read_text())
        print(f"{r.name}: accuracy {metrics['test_accuracy']:.3f}, macro-F1 {metrics['test_macro_f1']:.3f}")
        display(Image(filename=str(r / "confusion_matrix.png"), width=450))
        display(Image(filename=str(r / "predictions.png"), width=800))

## 9. Comparison: CNN from scratch vs transfer learning

**Extra seeds and training-data size (optional, several hours).** The main comparison repeats every model
with 3 seeds. The data-size experiment trains on 10%, 25% and 50% of the training set (`emotion7`) to see where
pretraining helps most; ResNet50 is left out of it to save GPU time. If Colab disconnects, re-run this cell:
finished runs are skipped.

In [ ]:
RUN_FULL_EXPERIMENTS = False

def train_once(model, labels, fraction, seed):
    name = f"fer2013_{model}_{labels}_f{fraction:g}_s{seed}"
    if (Path("results/runs") / name / "train_summary.json").exists():
        return
    !python -m src.training.train --model {model} --labels {labels} --train-fraction {fraction} --seed {seed}

if RUN_FULL_EXPERIMENTS:
    for seed in [1, 2]:  # seed 42 was trained above
        for model in ["scratch_cnn", "mobilenetv2", "resnet50"]:
            for labels in ["emotion7", "affect3"]:
                train_once(model, labels, 1.0, seed)
    for fraction in [0.1, 0.25, 0.5]:
        for seed in [42, 1, 2]:
            for model in ["scratch_cnn", "mobilenetv2"]:
                train_once(model, "emotion7", fraction, seed)
    !python -m src.evaluation.evaluate

Mean ± std over seeds for every model and configuration. Inference time is only comparable between runs
evaluated on the same hardware.

In [ ]:
!python -m src.evaluation.compare

summary = pd.read_csv("results/tables/comparison/summary.csv")
display(summary)
for figure in sorted(Path("results/figures/comparison").glob("*.png")):
    display(Image(filename=str(figure), width=600))

## 10. Save results

In [ ]:
# Download all figures, tables and run summaries as one zip (Colab only).
if IN_COLAB:
    !zip -qr results.zip results -x "*.keras"
    from google.colab import files
    files.download("results.zip")